# Modelos de lenguaje y RAG: pregúntale a la Constitución

**Sesión 9 · IA Intermedio · EstudIA 2026**

Hoy usamos modelos de lenguaje, los LLM, como los de los chats que ya conocen. Primero les preguntamos sin ayuda.
Después construimos un sistema RAG: para cada pregunta buscamos los artículos de la Constitución que sirven y se los
damos al modelo, que responde con ellos y dice de dónde sacó cada dato.

1. Un modelo de lenguaje, solo
2. Instrucciones y temperatura
3. La Constitución, en pedazos
4. Buscar por significado con Qdrant
5. RAG: buscar y responder con las fuentes
6. ¿Qué tan bien busca?
7. ¿Qué tan bien responde? Un modelo como juez
8. Cuatro modelos, el mismo RAG

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU. La celda de preparación te pide tu
clave: cópiala de la plataforma, en la actividad “Tu clave para los modelos de lenguaje”.

In [ ]:
# Preparación: librerías y tu clave. Ejecútala una vez (instalar tarda un momento).
!pip -q install "langchain-openai>=1.8" "langchain-qdrant>=1.2" "langchain-community>=0.4" "langchain-huggingface>=1.3" rank_bm25
import os, re, csv, json, time, getpass, urllib.request, urllib.error, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s09")
PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import openai

_CLAVE = os.environ.get("ESTUDIA_CLAVE") or getpass.getpass(
    "Pega tu clave (plataforma › Tu clave para los modelos de lenguaje › Copiar) y oprime Enter: ").strip()
cliente = openai.OpenAI(base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE, max_retries=3)
try:
    MODELOS = [m.id for m in cliente.models.list().data if getattr(m, "tipo", "chat") == "chat"]
    print("Listo. Modelos de lenguaje disponibles:", ", ".join(MODELOS))
except openai.AuthenticationError:
    print("La clave no es válida. Cópiala otra vez de la plataforma y vuelve a ejecutar esta celda.")
except openai.APIConnectionError:
    print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")

In [ ]:
# @title Funciones auxiliares: modelos de lenguaje, embeddings, Qdrant, buscar y medir
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

import torch
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
torch.set_num_threads(max(1, os.cpu_count() or 1))

def modelo_de_lenguaje(nombre, temperatura=0, max_tokens=400, **extra):
    """Un modelo de lenguaje de la plataforma, para usarlo con LangChain."""
    return ChatOpenAI(model=nombre, base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE, temperature=temperatura,
                      max_tokens=max_tokens, max_retries=6, timeout=120, **extra)

def preguntar(modelo, pregunta, sistema=None, mostrar=True, **opciones):
    """Una pregunta a un modelo, sin contexto. Devuelve la respuesta y muestra el tiempo y el costo."""
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    inicio = time.time()
    try:
        r = cliente.chat.completions.create(model=modelo, messages=mensajes, **opciones)
    except openai.APIStatusError as e:
        print("No se pudo:", (e.body or {}).get("message", e))
        return ""
    texto = r.choices[0].message.content or ""
    if mostrar:
        u = r.usage.model_dump()
        segundos = f"{time.time() - inicio:.1f}".replace(".", ",")
        costo = f"{u.get('costo_usd', 0):.6f}".replace(".", ",")
        print(f"{texto}\n  ({modelo} · {segundos} s · {u['prompt_tokens']} tokens de entrada y "
              f"{u['completion_tokens']} de salida · US$ {costo})")
    return texto

def cuanto_llevo():
    """Lo que llevas gastado hoy en la API de modelos de lenguaje."""
    req = urllib.request.Request(f"{PLATAFORMA}/v1/usage", headers={"Authorization": f"Bearer {_CLAVE}",
                                                                    "User-Agent": "EstudIA-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        u = json.loads(r.read())
    hoy, limite = f"{u['hoy_usd']:.4f}".replace(".", ","), f"{u['limite_diario_usd']:.2f}".replace(".", ",")
    print(f"Hoy llevas US$ {hoy} de US$ {limite} ({u['solicitudes_hoy']} solicitudes).")

_CON_PREFIJOS = []

class ConPrefijos(Embeddings):
    """Embeddings de la familia e5: “query: ” antes de cada pregunta y “passage: ” antes de cada texto (sesión 6).
    Guarda los vectores de las preguntas: preparar() los calcula todos de una vez."""
    def __init__(self, base, consulta="query: ", documento="passage: "):
        self.base, self.consulta, self.documento, self._cache = base, consulta, documento, {}
        _CON_PREFIJOS.append(self)
    def embed_documents(self, textos):
        return self.base.embed_documents([self.documento + t for t in textos])
    def embed_query(self, texto):
        if texto not in self._cache:
            self._cache[texto] = self.base.embed_query(self.consulta + texto)
        return self._cache[texto]
    def preparar(self, preguntas):
        nuevas = [p for p in dict.fromkeys(preguntas) if p not in self._cache]
        if nuevas:
            self._cache.update(zip(nuevas, self.base.embed_documents([self.consulta + p for p in nuevas])))

def embeddings_locales(nombre="intfloat/multilingual-e5-small"):
    """Un modelo de embeddings de Hugging Face que corre en Colab (sin GPU también)."""
    from langchain_huggingface import HuggingFaceEmbeddings
    return ConPrefijos(HuggingFaceEmbeddings(model_name=nombre, encode_kwargs={"normalize_embeddings": True,
                                                                              "batch_size": 32}))

def embeddings_api(nombre="intfloat/multilingual-e5-large"):
    """Un modelo de embeddings más grande, a través de la plataforma (cuesta, aunque muy poco)."""
    return ConPrefijos(OpenAIEmbeddings(model=nombre, base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE,
                                        check_embedding_ctx_length=False, chunk_size=128, max_retries=6))

def coleccion(documentos, embeddings, nombre="documentos"):
    """Una colección de Qdrant en la memoria de Colab con los vectores de los documentos."""
    inicio = time.time()
    c = QdrantVectorStore.from_documents(documentos, embeddings, location=":memory:", collection_name=nombre)
    print(f"{miles(len(documentos))} documentos en Qdrant en {time.time() - inicio:.0f} s.")
    return c

def buscar_hibrido(listas, k=5, c=60):
    """Combina varias listas de resultados (de buscadores distintos): gana lo que sale arriba en varias."""
    puntaje, docs = {}, {}
    for lista in listas:
        for puesto, d in enumerate(lista):
            clave = d.metadata.get("id", d.page_content[:80])
            puntaje[clave] = puntaje.get(clave, 0) + 1 / (c + puesto + 1)
            docs[clave] = d
    return [docs[i] for i in sorted(puntaje, key=lambda i: -puntaje[i])[:k]]

def preparar_preguntas(preguntas):
    for e in _CON_PREFIJOS:
        e.preparar(list(preguntas))

def medir_busqueda(buscar, preguntas, correctos, ks=(1, 3, 5), clave="id"):
    """En qué fracción de las preguntas el documento correcto sale entre los primeros k resultados."""
    preparar_preguntas(preguntas)
    aciertos = {k: 0 for k in ks}
    for p, ok in zip(preguntas, correctos):
        ids = [d.metadata.get(clave) for d in buscar(p)]
        ok = ok if isinstance(ok, (set, list, tuple)) else {ok}
        for k in ks:
            aciertos[k] += any(i in ok for i in ids[:k])
    return {f"en los primeros {k}": aciertos[k] / len(preguntas) for k in ks}

def puntos_busqueda(buscar, preguntas, correctos, k=5):
    """Como la plataforma: 1 punto si el correcto sale primero, 1/2 si sale segundo… 0 si no está entre los k primeros."""
    preparar_preguntas(preguntas)
    total = 0.0
    for p, ok in zip(preguntas, correctos):
        ids = [d.metadata.get("id") for d in buscar(p)][:k]
        total += 1 / (ids.index(ok) + 1) if ok in ids else 0.0
    return total / len(preguntas)

def responder_lote(cadena, buscar, preguntas, formato, k=5, simultaneas=4):
    """Busca y responde muchas preguntas a la vez. Devuelve (respuestas, documentos de cada pregunta)."""
    preparar_preguntas(preguntas)
    docs = [buscar(p)[:k] for p in preguntas]
    salidas = cadena.batch([{"contexto": formato(d), "pregunta": p} for d, p in zip(docs, preguntas)],
                           config={"max_concurrency": simultaneas}, return_exceptions=True)
    errores = [s for s in salidas if not isinstance(s, str)]
    if errores:
        print(f"{len(errores)} preguntas no se pudieron responder; por ejemplo: {str(errores[0])[:200]}")
    return [" ".join(s.split()) if isinstance(s, str) else "" for s in salidas], docs

JUEZ = ("Compara la respuesta de un sistema con la respuesta correcta a una pregunta. Escribe solo SI si dice lo "
        "mismo en lo esencial (aunque use otras palabras o agregue detalles) o NO si dice otra cosa, si le falta el "
        "dato principal o si dice que no sabe.")

def juez(preguntas, esperadas, respuestas, modelo="mistralai/mistral-small-3.2-24b-instruct"):
    """Un modelo de lenguaje como juez: ¿cada respuesta dice lo mismo que la esperada? Aproximado (y cuesta)."""
    cadena = (ChatPromptTemplate.from_messages([("system", JUEZ), ("human", "Pregunta: {p}\nRespuesta correcta: {e}\n"
                                                                          "Respuesta del sistema: {a}")])
              | modelo_de_lenguaje(modelo, max_tokens=5) | StrOutputParser())
    salidas = cadena.batch([{"p": p, "e": e, "a": a or "(sin respuesta)"} for p, e, a in zip(preguntas, esperadas, respuestas)],
                           config={"max_concurrency": 4}, return_exceptions=True)
    errores = [s for s in salidas if not isinstance(s, str)]
    if errores:
        print(f"El juez no pudo revisar {len(errores)} respuestas (cuentan como no aceptadas); por ejemplo: "
              f"{str(errores[0])[:200]}")
    return [isinstance(s, str) and s.strip().upper().startswith("S") for s in salidas]

---
## 1. Un modelo de lenguaje, solo

Un modelo de lenguaje predice la siguiente palabra (en realidad, el siguiente token) una y otra vez. Después de
entrenarlo con muchísimo texto, lo ajustan para seguir instrucciones. Le preguntamos a través de la plataforma, con
tu clave. Cada pregunta cuesta: el modelo cobra por los tokens que lee y los que escribe.

In [ ]:
MODELO = "mistralai/mistral-small-3.2-24b-instruct"
respuesta = preguntar(MODELO, "¿Qué edad hay que tener para ser senador en Colombia?")

In [ ]:
respuesta = preguntar(MODELO, "¿Qué dice el artículo 79 de la Constitución Política de Colombia?")

Los cuatro modelos con la misma pregunta: ¿responden lo mismo? ¿Cuánto tardan y cuánto cuestan?

In [ ]:
filas = []
for m in MODELOS:
    inicio = time.time()
    r = cliente.chat.completions.create(model=m, messages=[{"role": "user", "content":
        "¿Cuántos senadores elegidos por las comunidades indígenas tiene el Senado de Colombia? Responde en una frase."}],
        max_tokens=300)
    filas.append({"modelo": m, "respuesta": r.choices[0].message.content, "segundos": round(time.time() - inicio, 1),
                  "costo (US$)": r.usage.model_dump().get("costo_usd")})
CUATRO_MODELOS = pd.DataFrame(filas)
display(CUATRO_MODELOS.style.hide(axis="index"))
cuanto_llevo()

**Para pensar.**

- ¿Le creerías a cada respuesta? ¿Cómo sabrías si es cierta?
- ¿Qué pasaría con una pregunta sobre un documento que el modelo nunca vio, como el reglamento de tu empresa?

---
## 2. Instrucciones y temperatura

Una conversación con el modelo es una lista de mensajes. El mensaje de **sistema** le da instrucciones generales; el
del **usuario** trae la pregunta. La **temperatura** controla qué tanto varía: con 0 responde casi siempre lo mismo;
con 1, cambia.

In [ ]:
preguntar(MODELO, "¿Qué es una acción de tutela?", sistema="Responde en una sola frase, para un niño de 12 años.")
print()
for i in range(3):
    preguntar(MODELO, "Inventa un nombre para una tienda de café en Bogotá. Solo el nombre.", temperature=1.0)

---
## 3. La Constitución, en pedazos

Los 380 artículos de la Constitución Política de Colombia, como los publica la Secretaría del Senado. Cada artículo
es un documento, con su número, título y capítulo. Algunos artículos son muy largos: los partimos en pedazos que se
solapan un poco, y cada pedazo conserva el número de su artículo.

In [ ]:
articulos = json.load(open(descargar("constitucion.json"), encoding="utf-8"))
documentos = [Document(page_content=f"Artículo {a['articulo']}. {a['texto']}",
                       metadata={"articulo": a["articulo"], "titulo": a["titulo"], "capitulo": a["capitulo"]})
              for a in articulos]
largos = sorted(len(d.page_content) for d in documentos)
print(f"{len(documentos)} artículos; la mitad tiene menos de {miles(largos[len(largos) // 2])} caracteres y el más "
      f"largo, {miles(largos[-1])}.")

partidor = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
pedazos = partidor.split_documents(documentos)
print(f"Partidos en pedazos de máximo 1.500 caracteres: {len(pedazos)} pedazos.")
print("\nEl artículo 172 completo:\n", documentos[171].page_content)

---
## 4. Buscar por significado con Qdrant

Convertimos cada pedazo en un vector con el modelo de embeddings de la sesión 6 (e5 pequeño, que corre en Colab) y
guardamos los vectores en **Qdrant**, una base de datos de vectores; aquí vive en la memoria de Colab. Para buscar,
convertimos la pregunta en un vector y Qdrant devuelve los pedazos más cercanos.

In [ ]:
EMBEDDINGS = embeddings_locales("intfloat/multilingual-e5-small")
constitucion = coleccion(pedazos, EMBEDDINGS, "constitucion")
for d, s in constitucion.similarity_search_with_score("¿Qué edad hay que tener para ser senador?", k=3):
    print(f"{s:.3f}".replace(".", ","), f"Artículo {d.metadata['articulo']}:", d.page_content[:140], "…")

---
## 5. RAG: buscar y responder con las fuentes

RAG (*retrieval-augmented generation*, generación aumentada con búsqueda): buscamos los pedazos que sirven, los
ponemos en el mensaje como contexto y le pedimos al modelo que responda **solo** con ellos y cite el artículo. Si la
respuesta no está, que lo diga.

In [ ]:
INSTRUCCIONES = ("Respondes preguntas sobre la Constitución Política de Colombia usando solo los artículos del "
                 "contexto. Responde en una o dos frases. Cita entre corchetes el artículo de donde sacas cada dato, "
                 "por ejemplo [Artículo 172]. Si la respuesta no está en el contexto, di: “No lo encuentro en la "
                 "Constitución.”")
plantilla = ChatPromptTemplate.from_messages([("system", INSTRUCCIONES),
                                              ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}")])
cadena = plantilla | modelo_de_lenguaje(MODELO) | StrOutputParser()
buscador = constitucion.as_retriever(search_kwargs={"k": 4})

def contexto(docs):
    return "\n\n".join(d.page_content for d in docs)

def rag(pregunta):
    docs = buscador.invoke(pregunta)
    respuesta = cadena.invoke({"contexto": contexto(docs), "pregunta": pregunta})
    print(f"{pregunta}\n→ {respuesta}\n  (buscó los artículos {', '.join(str(d.metadata['articulo']) for d in docs)})\n")
    return respuesta

for p in ["¿Qué edad hay que tener para ser senador?", "¿Qué dice el artículo 79?",
          "¿Qué hago si me detienen de forma ilegal?", "¿Quién ganó el Mundial de fútbol de 2014?"]:
    rag(p)

**Para pensar.**

- Encontró “¿Qué dice el artículo 79?”, aunque un buscador por significado busca textos parecidos y un número no se parece a nada. ¿Por qué? Pista: mira con qué empieza cada pedazo.
- ¿Qué ganamos con que cite el artículo?

---
## 6. ¿Qué tan bien busca? Medir con preguntas que ya conocemos

Escribimos 28 preguntas comunes con el artículo que las responde. Medimos en cuántas el artículo correcto sale entre
los primeros 1, 3 o 5 resultados, con cuatro buscadores: BM25 (bolsa de palabras, sesión 5), e5 pequeño, e5 grande (a
través de la plataforma) y una combinación de BM25 con e5 grande.

In [ ]:
preguntas = pd.read_csv(descargar("preguntas_constitucion.csv"))
correctos = [set(int(x) for x in str(a).split(";")) for a in preguntas["articulos"]]

def sin_tildes(texto):
    import unicodedata
    t = unicodedata.normalize("NFKD", texto.lower())
    return re.findall(r"\w+", "".join(c for c in t if not unicodedata.combining(c)))

bm25 = BM25Retriever.from_documents(pedazos, k=10, preprocess_func=sin_tildes)
EMB_GRANDE = embeddings_api("intfloat/multilingual-e5-large")
constitucion_grande = coleccion(pedazos, EMB_GRANDE, "constitucion_grande")
buscadores = {
    "BM25": lambda p: bm25.invoke(p),
    "e5 pequeño": lambda p: constitucion.similarity_search(p, k=10),
    "e5 grande": lambda p: constitucion_grande.similarity_search(p, k=10),
    "BM25 + e5 grande": lambda p: buscar_hibrido([bm25.invoke(p), constitucion_grande.similarity_search(p, k=10)], k=10),
}
TABLA_BUSQUEDA = pd.DataFrame({n: medir_busqueda(b, preguntas["pregunta"], correctos, clave="articulo")
                               for n, b in buscadores.items()}).T
display(TABLA_BUSQUEDA.style.format(lambda x: pct(x)))
TABLA_BUSQUEDA.plot.barh(figsize=(8, 3.2), color=["#CB318E", "#808080", "#000000"]).invert_yaxis()
plt.xlim(0, 1); plt.xlabel("Fracción de preguntas con el artículo correcto entre los primeros k"); plt.tight_layout(); plt.show()
cuanto_llevo()

---
## 7. ¿Qué tan bien responde? Un modelo como juez

Con el mejor buscador, el RAG responde las 28 preguntas. Para no revisar todas a mano, otro modelo de lenguaje hace de
**juez**: compara cada respuesta con la que escribimos. Es una medida aproximada (el juez también se equivoca), pero
sirve para comparar. En la competencia, el juez es otro modelo (Jev), en la plataforma.

In [ ]:
mejor = TABLA_BUSQUEDA["en los primeros 5"].idxmax()
buscar = lambda p: buscadores[mejor](p)[:5]
formato = lambda docs: contexto(docs)
respuestas, docs = responder_lote(cadena, buscar, list(preguntas["pregunta"]), formato)
ok = juez(preguntas["pregunta"], preguntas["respuesta_esperada"], respuestas)
print(f"Buscador: {mejor}. El juez acepta {sum(ok)} de {len(ok)} respuestas ({pct(sum(ok) / len(ok))}).\n")
for p, e, r, bien in list(zip(preguntas["pregunta"], preguntas["respuesta_esperada"], respuestas, ok))[:28]:
    if not bien:
        print(f"✗ {p}\n   esperada: {e}\n   el RAG:   {r}\n")

**Para pensar.**

- ¿Las que el juez rechazó están mal de verdad, o el juez se equivocó?
- ¿Qué cambiarías primero: el buscador, las instrucciones o el modelo?

---
## 8. Cuatro modelos, el mismo RAG

El mismo buscador y las mismas instrucciones con los cuatro modelos: cuántas acepta el juez, cuánto tardan y cuánto
cuestan las 28 preguntas.

In [ ]:
filas = []
docs_28 = [buscar(p) for p in preguntas["pregunta"]]
for m in MODELOS:
    c = plantilla | modelo_de_lenguaje(m) | StrOutputParser()
    antes = json.loads(urllib.request.urlopen(urllib.request.Request(f"{PLATAFORMA}/v1/usage",
        headers={"Authorization": f"Bearer {_CLAVE}", "User-Agent": "EstudIA-notebook/1.0"})).read())["total_usd"]
    inicio = time.time()
    rs = c.batch([{"contexto": contexto(d), "pregunta": p} for d, p in zip(docs_28, preguntas["pregunta"])],
                 config={"max_concurrency": 4}, return_exceptions=True)
    rs = [r if isinstance(r, str) else "" for r in rs]
    segundos = time.time() - inicio
    despues = json.loads(urllib.request.urlopen(urllib.request.Request(f"{PLATAFORMA}/v1/usage",
        headers={"Authorization": f"Bearer {_CLAVE}", "User-Agent": "EstudIA-notebook/1.0"})).read())["total_usd"]
    bien = juez(preguntas["pregunta"], preguntas["respuesta_esperada"], rs)
    filas.append({"modelo": m, "acepta el juez": sum(bien) / len(bien), "segundos": round(segundos),
                  "costo de las 28 (US$)": round(despues - antes, 5)})
TABLA_MODELOS = pd.DataFrame(filas)
display(TABLA_MODELOS.style.format({"acepta el juez": lambda x: pct(x)}).hide(axis="index"))
cuanto_llevo()

**Para pensar.**

- ¿Cuál elegirías para un sistema que responde miles de preguntas al día? ¿Y para uno que responde pocas, pero donde equivocarse cuesta mucho?

---
**Lo que vimos.** Un modelo de lenguaje solo responde con lo que aprendió, y a veces inventa. Un RAG busca primero en
nuestros documentos y responde con ellos, citando la fuente. Qué tan bien busca y qué tan bien responde se mide por
separado, con preguntas cuya respuesta ya conocemos.